# CFB Line Accuracy Overview

How accurate are college football betting lines?
- Spread cover rates overall and by season
- Over/under accuracy
- Which factors predict line accuracy
- Where the edges are

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

ROOT = Path().resolve().parent
DB = duckdb.connect(str(ROOT / 'data/warehouse/ons.duckdb'), read_only=True)

plt.style.use('dark_background')
sns.set_palette('husl')

print('Connected. Tables:')
DB.execute("SHOW ALL TABLES").df()

## Overall accuracy

In [ ]:
overall = DB.execute("""
    SELECT
        count(*) as games,
        round(avg(case when spread_covered then 1.0 else 0.0 end) * 100, 2) as ats_pct,
        round(avg(case when ou_result = 'over' then 1.0 else 0.0 end) * 100, 2) as over_pct,
        sum(case when spread_push then 1 else 0 end) as spread_pushes,
        round(avg(abs(margin_vs_spread)), 2) as avg_spread_error,
        round(avg(abs(total_vs_ou)), 2) as avg_ou_error
    FROM main_marts.mart_cfbd_line_accuracy
    WHERE spread_result in ('covered','missed')
""").df()

print(overall.T)

## Cover rate by season

In [ ]:
by_season = DB.execute("""
    SELECT
        season,
        count(*) as games,
        round(avg(case when spread_covered then 1.0 else 0.0 end) * 100, 1) as ats_pct,
        round(avg(case when ou_result = 'over' then 1.0 else 0.0 end) * 100, 1) as over_pct
    FROM main_marts.mart_cfbd_line_accuracy
    WHERE spread_result in ('covered','missed')
    GROUP BY season ORDER BY season
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(by_season['season'], by_season['ats_pct'],
            color=['green' if v > 50 else 'red' for v in by_season['ats_pct']])
axes[0].axhline(50, color='white', linestyle='--', alpha=0.5)
axes[0].set_title('ATS Cover Rate by Season')
axes[0].set_ylabel('Cover %')
axes[0].set_ylim(45, 55)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

axes[1].bar(by_season['season'], by_season['over_pct'],
            color=['steelblue' if v > 50 else 'orange' for v in by_season['over_pct']])
axes[1].axhline(50, color='white', linestyle='--', alpha=0.5)
axes[1].set_title('Over Rate by Season')
axes[1].set_ylabel('Over %')
axes[1].set_ylim(45, 55)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()
display(by_season)

## Edge factors — where the money is

In [ ]:
edges = DB.execute("""
    SELECT *
    FROM main_marts.mart_cfbd_edge_factors
    WHERE games >= 50
    ORDER BY abs(ats_pct - 50) DESC
    LIMIT 20
""").df()

edges['ats_edge'] = edges['ats_pct'] - 50
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in edges['ats_edge']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(edges['factor_value'] + ' (' + edges['factor_type'] + ')',
               edges['ats_edge'], color=colors)
ax.axvline(0, color='white', linestyle='--', alpha=0.5)
ax.set_xlabel('ATS Edge vs 50% baseline')
ax.set_title('Top 20 ATS Edge Factors (deviation from 50%)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

display(edges[['factor_type','factor_value','games','ats_pct','over_pct']].head(20))

## SP+ upset alerts

In [ ]:
upset_alerts = DB.execute("""
    SELECT
        season, week, home_team, away_team,
        home_score, away_score, spread,
        home_sp_rating, away_sp_rating, sp_differential,
        spread_result, ou_bucket
    FROM main_marts.mart_cfbd_line_accuracy
    WHERE sp_upset_alert = true
      AND spread_result in ('covered','missed')
    ORDER BY season DESC, week DESC
    LIMIT 30
""").df()

cover_rate = upset_alerts['spread_result'].value_counts(normalize=True)['covered'] * 100
print(f'SP+ upset alert ATS cover rate: {cover_rate:.1f}% ({len(upset_alerts)} games)')

display(upset_alerts)